# Apollo Eval — Delta Notebook (EVAL-04, D-10)

Reads `eval/scores.jsonl` + `eval/runs.jsonl`, surfaces per-iteration deltas.

**Restart Kernel and Run All when you grade new sessions** (Jupyter cells cache).

Cells 1–5 are the canonical layout — do not delete. Add exploratory cells below cell 5.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
SCORES = Path('eval/scores.jsonl')
RUNS = Path('eval/runs.jsonl')

In [ ]:
scores = pd.read_json(SCORES, lines=True)
runs = pd.read_json(RUNS, lines=True)
# Last-write-wins per (run_id, pair_id, dim) — RESEARCH Pattern 1.
scores = scores.drop_duplicates(subset=['run_id','pair_id','dim'], keep='last')
print(f'{len(runs)} runs · {len(scores)} score records · '
      f"{runs['iteration'].sum() if 'iteration' in runs else 0} iteration-marked")

In [ ]:
# Plot 1 — Per-dim mean over runs (REQUIRED, EVAL-04 D-10)
means = scores.groupby(['run_id','dim'])['score'].mean().unstack()
means = means.reindex(runs.sort_values('created')['run_id'])
ax = means.plot(marker='o', figsize=(8,4))
ax.set_xticklabels([r[:8] for r in means.index], rotation=45)
ax.set_title('Mean score per dimension across runs')
ax.set_ylabel('mean score (1–5)')
plt.tight_layout(); plt.show()

In [ ]:
# Plot 2 — Per-pair call-response-fit trajectories (REQUIRED, EVAL-04 D-10)
fit = scores[scores.dim == 'fit']
pivot = fit.pivot_table(index='run_id', columns='pair_id', values='score')
pivot = pivot.reindex(runs.sort_values('created')['run_id'])
ax = pivot.plot(figsize=(10,5), legend=False, alpha=0.5)
ax.set_xticklabels([r[:8] for r in pivot.index], rotation=45)
ax.set_title('Per-pair fit trajectory'); ax.set_ylabel('fit score (1–5)')
plt.tight_layout(); plt.show()

In [ ]:
# Ship-gate (EVAL-05) — same logic as `python -m apollo.scripts.eval_ship_check`
from apollo.eval import check_ship_gate
passed, banner = check_ship_gate()
print(banner)
print('\n→', 'SHIP-READY' if passed else 'NOT YET — keep iterating')